# Data Wrangling 2.2

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2

import json

import csv

from datetime import datetime as dt

from IPython.display import display, HTML


from jellyfish import soundex, levenshtein_distance

from fuzzywuzzy import fuzz

from fuzzywuzzy import process as fuzz_process


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

# Lab: Data Cleansing - Fuzzy Logic, Soundex, Levenshtein Distances

## stage_3 tables will hold dirty data to allow us to see how to detect and clean it

In [ ]:
connection.rollback()

query = """

drop table if exists stage_3_customers;
drop table if exists stage_3_sales;
drop table if exists stage_3_line_items;



"""

cursor.execute(query)

connection.commit()



In [ ]:
#
# create staging tables with all varchar(100)
#

connection.rollback()

query = """


create table stage_3_customers (
  stage_id serial,
  customer_id varchar(100),
  first_name varchar(100),
  last_name varchar(100),
  street varchar(100),
  city varchar(100),
  state varchar(100),
  zip varchar(100),
  closest_store_id varchar(100),
  distance varchar(100)
);

create table stage_3_sales (
  stage_id serial,
  store_id varchar(100),
  sale_id varchar(100),
  customer_id varchar(100),
  sale_date varchar(100),
  total_amount varchar(100)
);

create table stage_3_line_items (
  stage_id serial,
  store_id varchar(100),
  sale_id varchar(100),
  line_item_id varchar(100),
  product_id varchar(100),
  quantity varchar(100)
);

"""

cursor.execute(query)

connection.commit()



In [ ]:
connection.rollback()

query = """

copy stage_3_customers (customer_id, first_name, last_name, street, city, state, zip, closest_store_id, distance)
from '/user/labs/week_07/dirty_data/dirty_customers.csv' delimiter ',' NULL '' csv header;

copy stage_3_sales (store_id, sale_id, customer_id, sale_date, total_amount)
from '/user/labs/week_07/dirty_data/dirty_sales.csv' delimiter ',' NULL '' csv header;

copy stage_3_line_items (store_id, sale_id, line_item_id, product_id, quantity)
from '/user/labs/week_07/dirty_data/dirty_line_items.csv' delimiter ',' NULL '' csv header;


"""

cursor.execute(query)

connection.commit()

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select * 
from stage_3_customers
order by stage_id;

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select * 
from stage_3_sales
order by stage_id;

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select * 
from stage_3_line_items
order by stage_id;

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## Soundex - starts with a letter with the phoenetic sound, followed by two digits for the phoentic sounds of the remaining consonants

In [ ]:
soundex("Berkeley")

In [ ]:
soundex("Berkely")

In [ ]:
soundex("Berklie")

In [ ]:
soundex("Barkly")

In [ ]:
soundex("Verkeley")

In [ ]:
soundex("there")

In [ ]:
soundex("their")

In [ ]:
soundex("Phoenix")

In [ ]:
soundex("fenix")

## Postgres also has a soundex function

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select first_name,
       soundex(first_name) as soundex_first_name,
       last_name,
       soundex(last_name) as soundex_last_name,
       city,
       soundex(city) as soundex_city
from stage_3_customers
order by stage_id;

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## Levenshtein Distance - distance is the number of character insertions, character deletions, and character changes to make the strings match

In [ ]:
levenshtein_distance("Berkeley", "Berkely")

In [ ]:
levenshtein_distance("Berkeley", "Berklie")

In [ ]:
levenshtein_distance("Berkeley", "Verkeley")

In [ ]:
levenshtein_distance("apples", "oranges")

## Using Levenshtein Distances to measure string differences in string kernels in machine learning 

In [ ]:
dna_strand_1 = "CCT CTT TGC ACT CGG ATC GTA CGC TAT TCT ATG ATT ACA CGG TTG CGA TCC ATA"

dna_strand_2 = "TCC CTT GGG GAA TAT ACA CGC TGG CTT ACT CGA ATT TGA CTC GTA CTC GCC ATC"

levenshtein_distance(dna_strand_1, dna_strand_2)

## Postgres also has a Levenshtein Distance function

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select first_name,
       last_name,
       levenshtein(first_name, last_name)
from stage_3_customers
where first_name is not null and last_name is not null
order by stage_id;

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## Fuzzy Logic - 100 is perfect match

In [ ]:
fuzz.ratio("Berkeley", "Berkeley")

In [ ]:
fuzz.ratio("Berkeley", "Berkely")

In [ ]:
fuzz.ratio("Berkeley", "Verkeley")

In [ ]:
fuzz.ratio("Go Bears", "Go Bears!!!")

In [ ]:
fuzz.partial_ratio("Go Bears", "Go Bears!!!")

In [ ]:
fuzz.ratio("Oski the Bear is our mascot", "Our mascot is the Bear Oski")

In [ ]:
fuzz.token_sort_ratio("Oski the Bear is our mascot", "Our mascot is the Bear Oski")

In [ ]:
fuzz.token_sort_ratio("Oski the Bear", "Our mascot is the Bear Oski")

In [ ]:
fuzz.token_sort_ratio("Go Bears!!!", "Our mascot is the Bear Oski")

In [ ]:
choices = ["Berkeley", "San Francisco", "San Jose", "Portland", "Seattle", "Los Angeles"]

In [ ]:
fuzz_process.extract("san fran", choices, limit=2)

In [ ]:
fuzz_process.extract("frisco", choices, limit=2)

In [ ]:
fuzz_process.extract("Dallas", choices, limit=2)

In [ ]:
fuzz_process.extractOne("san fran", choices)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select cu.stage_id,
       cu.city as stage_city,
       z.city as zip_codes_city
from stage_3_customers as cu
     join zip_codes as z
         on cu.zip = z.zip
where cu.city <> z.city
order by customer_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## Find the misspelled cities and explore soundex, Levenshtein distances, and fuzzy logic 

In [ ]:

connection.rollback()

query = """

select cu.stage_id,
       cu.city as stage_city,
       z.city as zip_codes_city
from stage_3_customers as cu
     join zip_codes as z
         on cu.zip = z.zip
where cu.city <> z.city
order by stage_id
;

"""
    
cursor.execute(query)

connection.rollback()
    
rows = cursor.fetchall()
    
for row in rows:
        print("---------------------------------------------------------")
        print("Wrong:", row[1], "soundex", soundex(row[1]))
        print("Right:", row[2], "soundex", soundex(row[2]))
        print("Levenshtein Distance:", levenshtein_distance(row[1], row[2]))
        print("Fuzzy: ratio:", fuzz.ratio(row[1], row[2]))
        print("Fuzzy: partial ratio:", fuzz.partial_ratio(row[1], row[2]))
        print("Fuzzy: token sort ratio:", fuzz.partial_ratio(row[1], row[2]))
        

In [ ]:

connection.rollback()

query = """

select distinct city
from cities
order by 1
;

"""
    
cursor.execute(query)

connection.rollback()
    
rows = cursor.fetchall()
    
city_list = []
    
for row in rows:
        city_list.append(row[0])
        
print(city_list[:100])

In [ ]:

connection.rollback()

query = """

select cu.stage_id,
       cu.city as stage_city,
       z.city as zip_codes_city
from stage_3_customers as cu
     join zip_codes as z
         on cu.zip = z.zip
where cu.city <> z.city
order by customer_id
;

"""
    
cursor.execute(query)

connection.rollback()
    
rows = cursor.fetchall()
    
for row in rows:
        print("---------------------------------------------------------")
        print("Wrong:", row[1], "soundex", soundex(row[1]))
        print("Right:", row[2], "soundex", soundex(row[2]))
        print("Levenshtein Distance:", levenshtein_distance(row[1], row[2]))
        print("Fuzzy top 5 choices:", fuzz_process.extract(row[1], city_list, limit=5))
        

## You try it - Repeat for bad customer first names

# Lab: Data Cleansing - Dedup (Removing Duplicates)

## Find duplicates in stage_3_customers

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select cu.customer_id,
       cu.first_name,
       cu.last_name,
       cu.street,
       cu.city,
       cu.state,
       cu.zip,
       cu.closest_store_id,
       cu.distance,
       count(*) number_of_duplicates
from stage_3_customers as cu
group by cu.customer_id, cu.first_name, cu.last_name, cu.street, 
         cu.city, cu.state, cu.zip, cu.closest_store_id, cu.distance
having count(*) > 1
order by customer_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

with a as (

    select cu.customer_id,
           cu.first_name,
           cu.last_name,
           cu.street,
           cu.city,
           cu.state,
           cu.zip,
           cu.closest_store_id,
           cu.distance
    from stage_3_customers as cu
    group by cu.customer_id, cu.first_name, cu.last_name, cu.street, 
             cu.city, cu.state, cu.zip, cu.closest_store_id, cu.distance
    having count(*) > 1

    )

select *
from stage_3_customers
where customer_id in (select customer_id from a)
order by stage_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - Find the duplicate sales 

# Lab: Data Cleansing - Missing Values

## Find missing cities in stage_3_customers

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stage_3_customers
where city is null
order by customer_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - Find the missing customer first names  

# Lab: Data Cleansing - Outliers

## Find outliers on total_amount in state_3_sales

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stage_3_sales
where total_amount::numeric > 100
order by store_id, sale_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stage_3_sales
where total_amount::numeric > 200
order by store_id, sale_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - Find the outliers for line item quantity